# Mapping CUAD annotated spans to chunks (#24)

Labels every chunk from #23 with the CUAD categories whose annotated spans overlap it.
`label_chunks(chunks, categories)` in `src/ingest/label_chunks.py` does the work. This
notebook holds examples and checks for correctness, then saves the label files.

Every label comes from span overlap: a span's `[answer_start, answer_end)` is intersected
with each chunk's `[start_char, end_char)`. Contract-level presence never sets a label. It
is only used afterwards, as a check (section 9).

## Decisions

**Spans that cross a chunk boundary, and chunks that overlap** (`rule="any_overlap"`, the
default, chosen by the team)

- A chunk is positive for a category when it overlaps any of that category's spans by at
  least one character.
- A span cut by a chunk boundary marks every chunk it touches.
- Chunks are labelled independently. A span inside the 128-token overlap between two chunks
  marks both, so chunk-level counts include duplicates. Predictions must be merged back to
  contract level before scoring (#22).
- A chunk that holds only a small piece of a span that is whole in a neighbouring chunk is
  positive too. Section 8 shows how many labels that affects.
- `rule="contained_first"` is the alternative: a chunk is positive only if it fully contains
  a span, and a span that no chunk contains marks every chunk it touches.

**What a label means**

| Label | Meaning |
|---|---|
| `True` | a span marks this chunk under the rule (by default, it overlaps the chunk) |
| `False` | a genuine negative: the contract has no span for the category (status `absent`), or none of its spans mark this chunk |
| missing (`<NA>`) | not usable as a negative: the category was dropped from this split, or the contract/category has a span that is `unplaced` or `misaligned` |

**Status of each contract/category**

| Status | Meaning |
|---|---|
| `absent` | CUAD has no span |
| `placed` | every span overlaps a chunk and its text matches the chunk text |
| `unplaced` | a span overlaps no chunk |
| `misaligned` | a span's text differs from the chunk text at its offsets (or the span is empty) |

A span that is `unplaced` or `misaligned` marks nothing, and the `False` labels for that
contract/category become missing, so a lost clause is never trained on as a negative.

## Offset and text alignment assumptions

1. Every offset is a Python string index into the contract's original `context`
   (`paragraphs[0].context`). The chunker received that same string with no cleaning, so
   spans and chunks share one coordinate system.
2. Intervals are half-open. A span covers `[answer_start, answer_start + len(text))` and a
   chunk covers `[start_char, end_char)`, so a span that only touches a chunk edge does not
   overlap it.
3. Chunk offsets increase strictly in both `start_char` and `end_char` within a contract,
   so the chunks a span overlaps are a consecutive run. `label_chunks` raises an error if
   this ever stops being true.
4. Chunks leave out only whitespace: leading, trailing, or between chunks. CUAD spans have
   no leading or trailing whitespace, so a valid span always overlaps a chunk.
5. The span text must equal the chunk text over the overlap. That is how a mismatch
   between offsets and text is detected, and it needs the `text` column of `chunks.parquet`.

In [2]:
# Loading the data and the labeller
import sys
sys.path.insert(0, "..")          # so `src` can be imported from the notebooks folder

import numpy as np
import pandas as pd

from src.ingest.label_chunks import META_COLUMNS, label_chunks

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

chunks = pd.read_parquet("../data/processed/chunks.parquet")
categories = pd.read_json("../data/processed/contract_categories.jsonl", lines=True)

print("chunks:", len(chunks), "| contract/category rows:", len(categories))
print("categories per split:", categories.groupby("split")["category"].nunique().to_dict())

chunks: 17079 | contract/category rows: 18054
categories per split: {'test': 41, 'train': 34}


In [3]:
# Label every chunk
result = label_chunks(chunks, categories)                  # rule="any_overlap"
chunk_labels, overlaps, span_audit, pair_status = result

label_columns = [c for c in chunk_labels.columns if c not in META_COLUMNS]
print("chunk_labels: ", chunk_labels.shape, f"({len(label_columns)} category columns)")
print("span_overlaps:", len(overlaps), "(span, chunk) intersections")
print("span_audit:   ", len(span_audit), "spans")
print("pair_status:  ", len(pair_status), "contract/category pairs")

chunk_labels:  (17079, 48) (41 category columns)
span_overlaps: 19007 (span, chunk) intersections
span_audit:    13544 spans
pair_status:   18054 contract/category pairs


In [4]:
# 1. Do the offsets point at the annotated words?
ends_match = all(a["answer_end"] == a["answer_start"] + len(a["text"])
                 for answers in categories["answers"] for a in answers)
print("answer_end == answer_start + len(text) for every span:", ends_match)
print("span text equals chunk text over every overlap:       ", bool(overlaps["text_matches"].all()))
print("spans that overlap no chunk:                          ", int((span_audit["n_chunks"] == 0).sum()))
print()
print("span status:", span_audit["status"].value_counts().to_dict())
print()

# kind: how the span sits against the chunks
#   contained   whole in exactly one chunk (it may also reach a few characters into a neighbour)
#   in_overlap  whole in two or more chunks, because it lies in the overlap between them
#   crossing    whole in no chunk: a boundary cuts it
kinds = span_audit["kind"].value_counts().rename("spans").to_frame()
kinds["pct"] = (100 * kinds["spans"] / kinds["spans"].sum()).round(1)
kinds

answer_end == answer_start + len(text) for every span: True
span text equals chunk text over every overlap:        True
spans that overlap no chunk:                           0

span status: {'placed': 13544}



,spans,pct
kind,,
contained,11502,84.9
in_overlap,1752,12.9
crossing,290,2.1


In [5]:
# Helpers for the examples below
def get_span(contract_id, category, span_index):
    row = categories[(categories["contract_id"] == contract_id) & (categories["category"] == category)]
    return row.iloc[0]["answers"][span_index]


def describe_span(contract_id, category, span_index):
    audit = span_audit[(span_audit["contract_id"] == contract_id) & (span_audit["category"] == category)
                       & (span_audit["span_index"] == span_index)].iloc[0]
    text = " ".join(get_span(contract_id, category, span_index)["text"].split())
    print(contract_id[:70])
    print(f"category: {category} | span [{audit['answer_start']}, {audit['answer_end']}) "
          f"= {audit['span_chars']} characters | kind: {audit['kind']}")
    print("span text:", text[:200] + (" ..." if len(text) > 200 else ""))
    print()


def show_span(contract_id, category, span_index, context=1, labels=None):
    """The chunks around one span: the label for its category and how the span overlaps each."""
    labels = chunk_labels if labels is None else labels
    overlap = overlaps[(overlaps["contract_id"] == contract_id) & (overlaps["category"] == category)
                       & (overlaps["span_index"] == span_index)]
    lo, hi = overlap["chunk_index"].min() - context, overlap["chunk_index"].max() + context
    rows = labels[(labels["contract_id"] == contract_id) & labels["chunk_index"].between(lo, hi)]
    table = rows[["chunk_index", "start_char", "end_char", category]].rename(columns={category: "label"})
    table = table.merge(overlap[["chunk_index", "overlap_chars", "share_of_span", "fully_contained"]],
                        on="chunk_index", how="left")
    table["overlap_chars"] = table["overlap_chars"].fillna(0).astype(int)
    table["share_of_span"] = table["share_of_span"].round(2)
    table["fully_contained"] = table["fully_contained"].eq(True)
    return table.reset_index(drop=True)


# The examples use contract/category pairs with a single span, so every label in a table
# comes from the span being shown and not from another span of the same category.
audit = span_audit.merge(pair_status[["contract_id", "category", "n_spans"]], on=["contract_id", "category"])

In [6]:
# 2. A span fully inside one chunk
chunks_per_contract = chunks.groupby("contract_id").size()
pick = audit[(audit["split"] == "train") & (audit["n_spans"] == 1) & (audit["kind"] == "contained")
             & (audit["n_chunks"] == 1) & audit["span_chars"].between(60, 250)
             & (audit["answer_start"] > 3000)
             & (audit["contract_id"].map(chunks_per_contract) >= 5)].iloc[0]

describe_span(pick["contract_id"], pick["category"], pick["span_index"])
show_span(pick["contract_id"], pick["category"], pick["span_index"])

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
category: Governing Law | span [52061, 52151) = 90 characters | kind: contained
span text: This Agreement is to be construed according to the laws of the State of Illinois.



,chunk_index,start_char,end_char,label,overlap_chars,share_of_span,fully_contained
0,57,50500,51990,False,0,NaN,False
1,58,51649,52935,True,90,1.0,True
2,59,52610,53860,False,0,NaN,False


In [7]:
# 3. A span cut by a chunk boundary: no chunk holds all of it
pick = (audit[(audit["split"] == "train") & (audit["n_spans"] == 1) & (audit["kind"] == "crossing")]
        .sort_values("span_chars").iloc[0])
cid, cat, k = pick["contract_id"], pick["category"], pick["span_index"]
describe_span(cid, cat, k)

touched = overlaps[(overlaps["contract_id"] == cid) & (overlaps["category"] == cat) & (overlaps["span_index"] == k)]
first, last = touched.iloc[0], touched.iloc[-1]
raw = get_span(cid, cat, k)["text"]
print(f"chunk {first['chunk_index']} ends inside the clause:  ...", " ".join(raw[:first["overlap_chars"]].split())[-90:])
print(f"chunk {last['chunk_index']} starts inside the clause:", " ".join(raw[-last["overlap_chars"]:].split())[:90], "...")
print()
show_span(cid, cat, k)

NETGEAR,INC_04_21_2003-EX-10.16-AMENDMENT TO THE DISTRIBUTOR AGREEMENT
category: Document Name | span [3255, 3540) = 285 characters | kind: crossing
span text: AMENDMENT TO THE DISTRIBUTOR AGREEMENT BETWEEN INGRAM MICRO AND NETGEAR

chunk 2 ends inside the clause:  ... AMENDMENT TO THE DISTRIBUTOR AGREEMENT BETWEEN INGRAM MICRO
chunk 3 starts inside the clause: TO THE DISTRIBUTOR AGREEMENT BETWEEN INGRAM MICRO AND NETGEAR ...



,chunk_index,start_char,end_char,label,overlap_chars,share_of_span,fully_contained
0,1,1540,2824,False,0,NaN,False
1,2,2642,3460,True,205,0.72,False
2,3,3300,4706,True,240,0.84,False
3,4,4247,5917,False,0,NaN,False


In [8]:
# 4. Chunks overlap: a span inside the overlap marks both chunks
pick = audit[(audit["split"] == "train") & (audit["n_spans"] == 1) & (audit["kind"] == "in_overlap")
             & audit["span_chars"].between(60, 250)].iloc[0]

describe_span(pick["contract_id"], pick["category"], pick["span_index"])
show_span(pick["contract_id"], pick["category"], pick["span_index"])

ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT
category: Governing Law | span [19173, 19265) = 92 characters | kind: in_overlap
span text: This Agreement shall be governed and construed according to the laws of the State of Kansas.



,chunk_index,start_char,end_char,label,overlap_chars,share_of_span,fully_contained
0,10,15480,17726,False,0,NaN,False
1,11,17209,19363,True,92,1.0,True
2,12,18768,21227,True,92,1.0,True
3,13,20605,23047,False,0,NaN,False


In [9]:
# 5. A span that is whole in one chunk but has a small tail in the next
result_alt = label_chunks(chunks, categories, rule="contained_first")

candidates = overlaps.merge(audit[["contract_id", "category", "span_index", "n_containing", "n_spans"]],
                            on=["contract_id", "category", "span_index"])
pick = candidates[(candidates["split"] == "train") & (candidates["n_spans"] == 1)
                  & (candidates["n_containing"] == 1)
                  & ~candidates["fully_contained"] & candidates["share_of_span"].between(0.05, 0.3)
                  & candidates["span_chars"].between(60, 250)].iloc[0]
cid, cat, k = pick["contract_id"], pick["category"], pick["span_index"]
describe_span(cid, cat, k)

table = show_span(cid, cat, k).rename(columns={"label": "any_overlap"})
table["contained_first"] = show_span(cid, cat, k, labels=result_alt.chunk_labels)["label"].to_numpy()
table[["chunk_index", "start_char", "end_char", "any_overlap", "contained_first",
       "overlap_chars", "share_of_span", "fully_contained"]]

KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSULTING AGREEMENT
category: Expiration Date | span [7676, 7781) = 105 characters | kind: contained
span text: This Agreement will commence on the Effective Date and will continue until termination as provided below.



,chunk_index,start_char,end_char,any_overlap,contained_first,overlap_chars,share_of_span,fully_contained
0,2,3856,6501,False,False,0,NaN,False
1,3,5841,8365,True,True,105,1.00,True
2,4,7775,10347,True,False,6,0.06,False
3,5,9646,12273,False,False,0,NaN,False


In [10]:
# 6. A contract/category with no span, next to a category that was never labelled
dropped = "Source Code Escrow"          # dropped from train in the data prep, kept in test
pick = pair_status[(pair_status["split"] == "train") & (pair_status["status"] == "absent")
                   & pair_status["contract_id"].map(chunks_per_contract).between(5, 8)].iloc[0]
print(pick["contract_id"][:70])
print(f"category: {pick['category']} | spans: {pick['n_spans']} | status: {pick['status']}")
print()

rows = chunk_labels[chunk_labels["contract_id"] == pick["contract_id"]]
table = rows[["chunk_index", "start_char", "end_char", pick["category"], dropped]]
table.columns = ["chunk_index", "start_char", "end_char", "absent: " + pick["category"], "never labelled: " + dropped]
table.reset_index(drop=True)

VEONEER,INC_02_21_2020-EX-10.11-JOINT VENTURE AGREEMENT
category: Renewal Term | spans: 0 | status: absent



,chunk_index,start_char,end_char,absent: Renewal Term,never labelled: Source Code Escrow
0,0,0,1824,False,<NA>
1,1,1404,3332,False,<NA>
2,2,2876,4891,False,<NA>
3,3,4334,6469,False,<NA>
4,4,5977,7928,False,<NA>
5,5,7443,8257,False,<NA>


In [11]:
# The never-labelled category is missing on every train chunk but labelled on test chunks.
pd.crosstab(chunk_labels["split"], chunk_labels[dropped].astype("object").fillna("missing"))

Source Code Escrow,False,True,missing
split,,,
test,2896,2,0
train,0,0,14181


In [12]:
# 7. Spans that cannot be placed or do not align. CUAD has none, so this is a toy contract.
toy_text = "".join(chr(97 + i % 26) for i in range(100))
toy_chunks = pd.DataFrame([
    {"split": "train", "contract_id": "toy", "chunk_id": f"toy__{i}", "chunk_index": i,
     "start_char": s, "end_char": e, "n_tokens": 1, "text": toy_text[s:e]}
    for i, (s, e) in enumerate([(0, 40), (30, 70), (60, 95)])
])


def toy_row(category, start, text):
    answers = [{"text": text, "answer_start": start}] if text else []
    return {"split": "train", "contract_id": "toy", "category": category,
            "present": bool(answers), "answers": answers}


toy_categories = pd.DataFrame([
    toy_row("Placed", 10, toy_text[10:20]),
    toy_row("Absent", 0, ""),
    toy_row("Unplaced", 96, toy_text[96:99]),            # starts after the last chunk ends
    toy_row("Misaligned", 10, "not the words at 10"),    # the offsets point at different text
])

toy = label_chunks(toy_chunks, toy_categories)
print(toy.pair_status[["category", "status"]].to_string(index=False))
print()
toy.chunk_labels.drop(columns=["split", "contract_id", "chunk_id", "start_char", "n_tokens"])

  category     status
    Placed     placed
    Absent     absent
  Unplaced   unplaced
Misaligned misaligned



,chunk_index,end_char,Absent,Misaligned,Placed,Unplaced
0,0,40,False,<NA>,True,<NA>
1,1,70,False,<NA>,False,<NA>
2,2,95,False,<NA>,False,<NA>


In [13]:
# 8. How much does the boundary rule matter?
rows = []
for name, res in [("any_overlap", result), ("contained_first", result_alt)]:
    marked = res.span_overlaps[res.span_overlaps["marks_chunk"]]
    cells = int(sum(res.chunk_labels[c].fillna(False).sum() for c in label_columns))
    rows.append({"rule": name,
                 "span/chunk overlaps marked": len(marked),
                 "positive chunk/category labels": cells,
                 "spans with a positive chunk": marked.groupby(["contract_id", "category", "span_index"]).ngroups})
comparison = pd.DataFrame(rows)

# A sliver: the chunk holds part of a span that is whole in another chunk.
whole_elsewhere = overlaps.groupby(["contract_id", "category", "span_index"])["fully_contained"].transform("any")
slivers = overlaps[~overlaps["fully_contained"] & whole_elsewhere]
print(f"slivers: {len(slivers):,} of {len(overlaps):,} overlaps ({100 * len(slivers) / len(overlaps):.1f}%)")
print(f"a sliver chunk holds a median {100 * slivers['share_of_span'].median():.0f}% of its span; any_overlap labels them positive")
print()
comparison

slivers: 3,382 of 19,007 overlaps (17.8%)
a sliver chunk holds a median 43% of its span; any_overlap labels them positive



,rule,span/chunk overlaps marked,positive chunk/category labels,spans with a positive chunk
0,any_overlap,19007,12807,13544
1,contained_first,15625,10681,13544


In [14]:
# 9. Check every label against the spans
cats = np.array(label_columns)
cell_chunk, cell_cat = np.nonzero(chunk_labels[label_columns].fillna(False).to_numpy())
true_cells = set(zip(chunk_labels["contract_id"].to_numpy()[cell_chunk],
                     chunk_labels["chunk_index"].to_numpy()[cell_chunk], cats[cell_cat]))
marked = overlaps[overlaps["marks_chunk"]]
marked_cells = set(zip(marked["contract_id"], marked["chunk_index"], marked["category"]))
assert true_cells == marked_cells, "a label is not backed by a span overlap, or the reverse"
print(f"{len(true_cells):,} True labels, each from a span overlap, and every marking span set a label")

# Contract level, rebuilt from the chunk labels, must agree with CUAD's own `present`.
filled = chunk_labels[label_columns].fillna(False).astype(bool)
filled["contract_id"] = chunk_labels["contract_id"].to_numpy()
rebuilt = (filled.groupby("contract_id").any().reset_index()
           .melt(id_vars="contract_id", var_name="category", value_name="has_positive_chunk"))
check = categories[["contract_id", "category", "present"]].merge(rebuilt, on=["contract_id", "category"])
assert len(check) == len(categories)
assert (check["has_positive_chunk"] == check["present"]).all()
print(f"contract level agrees with CUAD presence for all {len(check):,} contract/category pairs")

assert pair_status.loc[pair_status["status"] == "absent", "n_positive_chunks"].eq(0).all()
print("no absent contract/category has a positive chunk")

# Missing labels: exactly the categories dropped from train, nowhere else.
in_train = set(categories.loc[categories["split"] == "train", "category"])
dropped_cols = sorted(set(label_columns) - in_train)
train_rows = chunk_labels["split"] == "train"
assert all(chunk_labels.loc[train_rows, c].isna().all() for c in dropped_cols)
assert not chunk_labels.loc[~train_rows, label_columns].isna().any().any()
assert not chunk_labels.loc[train_rows, [c for c in label_columns if c in in_train]].isna().any().any()
print(f"missing labels: only the {len(dropped_cols)} categories dropped from train, on train chunks")

12,807 True labels, each from a span overlap, and every marking span set a label
contract level agrees with CUAD presence for all 18,054 contract/category pairs
no absent contract/category has a positive chunk
missing labels: only the 7 categories dropped from train, on train chunks


In [15]:
# 10. Positive chunks per category
train_rows = chunk_labels["split"] == "train"
labels_train, labels_test = chunk_labels.loc[train_rows, label_columns], chunk_labels.loc[~train_rows, label_columns]
summary = pd.DataFrame({
    "train_positive_chunks": labels_train.sum(min_count=1),
    "train_pct_of_chunks": (100 * labels_train.mean()).round(2),
    "test_positive_chunks": labels_test.sum(min_count=1),
    "test_pct_of_chunks": (100 * labels_test.mean()).round(2),
}).sort_values("train_positive_chunks", ascending=False)
summary

,train_positive_chunks,train_pct_of_chunks,test_positive_chunks,test_pct_of_chunks
License Grant,701,4.94,152,5.24
Cap On Liability,640,4.51,122,4.21
Anti-Assignment,588,4.15,136,4.69
Parties,548,3.86,147,5.07
Expiration Date,525,3.7,111,3.83
Governing Law,521,3.67,138,4.76
Audit Rights,486,3.43,94,3.24
Document Name,447,3.15,113,3.9
Post-Termination Services,418,2.95,85,2.93
Effective Date,408,2.88,99,3.42


In [16]:
# 11. Save the label files
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

files = {"chunk_labels": chunk_labels, "span_overlaps": overlaps,
         "span_audit": span_audit, "pair_status": pair_status}
for name, frame in files.items():
    path = out_dir / f"{name}.parquet"
    frame.to_parquet(path, index=False)
    print(f"Saved {len(frame):,} rows to {path} ({path.stat().st_size / 1e6:.1f} MB)")

Saved 17,079 rows to ../data/processed/chunk_labels.parquet (0.5 MB)
Saved 19,007 rows to ../data/processed/span_overlaps.parquet (0.3 MB)
Saved 13,544 rows to ../data/processed/span_audit.parquet (0.2 MB)
Saved 18,054 rows to ../data/processed/pair_status.parquet (0.1 MB)


## Notes

**The files.** `chunk_labels.parquet` is the training table: one row per chunk and one
nullable boolean column per category. `span_audit.parquet` and `pair_status.parquet` say
where every span landed and whether each contract/category can be trusted.
`span_overlaps.parquet` holds every span/chunk intersection with its overlap size, so a
different rule can be tried without redoing the intersection. None of them contains
contract text.

**Results on this data.** All 13,544 spans in the saved files are `placed`, and every span
text equals the chunk text it overlaps. 290 spans (2.1%) are cut by a chunk boundary and
are whole in no chunk. The `unplaced` and `misaligned` handling is exercised only by the
toy contract in section 7, because CUAD has no such case. It protects against a future
chunker or data change.

**The boundary rule.** The team chose plain overlap, so a chunk is positive if it touches a
span at all. 17.8% of the marked span/chunk pairs are slivers: the chunk holds only part of
a span that is whole in a neighbour (a median 43% of it). They are labelled positive, which
adds some noise at chunk boundaries. `contained_first` would drop them and still label the
290 spans that no chunk contains. Switch with `label_chunks(..., rule="contained_first")`;
section 8 compares the two.

**Duplicates.** A span in the overlap between two chunks marks both, so chunk-level
counts and metrics include it twice. Merge chunk predictions back to contract level
before scoring (#22).

**Dropped categories.** The 7 categories dropped from train are missing on every train
chunk, and are labelled on test chunks. Decide before evaluation whether test scoring uses
only the 34 trained categories.

**Alignment check scope.** Span text is compared with chunk text, not the full contract.
That covers every character a chunk holds. Whitespace between chunks is the only text not
compared, and spans contain none at their edges.